<a href="https://colab.research.google.com/github/akshay-aiml/LangChain_LangGraph/blob/main/Hybrid_Search_Reranker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q  langchain langchain-community faiss-cpu rank_bm25 sentence-transformers transformers  langchain-text-splitters langchain-core langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 6.1 MB/s eta 0:00:00


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder
import numpy as np
from langchain_groq import ChatGroq

In [ ]:
from google.colab import userdata

api_key = userdata.get("GROQ_API_KEY")
# Initialize LLM

llm = ChatGroq(
    api_key=api_key,
    model="llama-3.3-70b-versatile",
    temperature=0.2
)

In [ ]:
# 1. Prepare Documents

texts = [
    "LangChain is a framework for building LLM applications.",
    "Hybrid search combines sparse and dense retrieval.",
    "Re-ranking improves retrieval precision.",
    "FAISS is used for vector similarity search.",
]

In [ ]:
documents = [Document(page_content=t) for t in texts]

In [ ]:
documents

[Document(metadata={}, page_content='LangChain is a framework for building LLM applications.'),
 Document(metadata={}, page_content='Hybrid search combines sparse and dense retrieval.'),
 Document(metadata={}, page_content='Re-ranking improves retrieval precision.'),
 Document(metadata={}, page_content='FAISS is used for vector similarity search.')]

In [ ]:
# 2. Dense Retriever (FAISS)

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = FAISS.from_documents(documents, embedding_model)

/tmp/ipython-input-347/2197646952.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
tokenized_docs = [doc.page_content.split() for doc in documents]
bm25 = BM25Okapi(tokenized_docs)

In [ ]:
# 4. Hybrid Retrieval Function
# ---------------------------

def hybrid_retrieve(query, top_k=3):
    # Dense search
    dense_docs = vectorstore.similarity_search(query, k=top_k)

    # Sparse search
    tokenized_query = query.split()
    sparse_scores = bm25.get_scores(tokenized_query)

    # Get top sparse docs
    sparse_indices = np.argsort(sparse_scores)[::-1][:top_k]
    sparse_docs = [documents[i] for i in sparse_indices]

    # Combine
    combined_docs = list({doc.page_content: doc for doc in dense_docs + sparse_docs}.values())
    return combined_docs

In [ ]:
# 5. Re-ranker
# ---------------------------
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank(query, docs, top_n=2):
    pairs = [(query, doc.page_content) for doc in docs]
    scores = reranker.predict(pairs)

    ranked = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)
    return [doc for doc, _ in ranked[:top_n]]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# 6. Full Pipeline
# ---------------------------
def retrieve_and_rerank(query):
    hybrid_docs = hybrid_retrieve(query)
    final_docs = rerank(query, hybrid_docs)
    return final_docs

In [ ]:
# Test
query = "What is hybrid search?"
results = retrieve_and_rerank(query)

for doc in results:
    print(doc.page_content)

Hybrid search combines sparse and dense retrieval.
FAISS is used for vector similarity search.
